# OK / AVVIK – visuell kvalitetskontroll

![Visuell kvalitetskontroll av støpehjul](figures/industri_kvalitetskontroll.png)

Fra Kaggle-settet: **OK** (hel kant) til venstre, **AVVIK** (støpefeil i kanten) til høyre. Dette er samme type bilde som du skal velge ut og merke.

**Navn (alene eller gruppe inntil 3):** ______

Les oppgaven i `Arbeidskrav_1_Visuell_kvalitetskontroll.docx` først.  
**Alt skrives i denne notebooken** (`quality_control.ipynb`) – ikke lag `Besvarelse_Etternavn.ipynb`.  
Kjør cellene i rekkefølge. Frist: **11. oktober 2026**.

Koding: `0 = OK`, `1 = AVVIK`. Ikke CNN. Ikke Adam.

**Bilder:** last ned Kaggle-settet og bruk mappa `casting_512x512` (`ok_front` → `data/ok/`, `def_front` → `data/avvik/`). Velg **6 andre filer** til `field_cases/`. Ikke bruk `casting_data` eller train/test-mappene.

## 1. Før trening (Del C, punkt 1)

**TODO – skriv her før du trener.** Hva skiller modellen, og hvilken feil er dyrest her?

Modellen skiller **______** og **______**.  
Koding: `0 = OK`, `1 = AVVIK`.  
Mest alvorlige feil her: **falsk OK**, fordi **______**.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

PROSJEKT = Path(".").resolve()
DATA_OK = PROSJEKT / "data" / "ok"
DATA_AVVIK = PROSJEKT / "data" / "avvik"
FELT = PROSJEKT / "field_cases"
MODELS = PROSJEKT / "models"
FIGURES = PROSJEKT / "figures"

IMAGE_SIZE = 28
KLASSER = ["ok", "avvik"]  # indeks 0 = OK, indeks 1 = AVVIK
TYPER = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def si(*linjer):
    # Skriv en tydelig forklaring i output under cellen.
    print()
    print("-" * 64)
    for linje in linjer:
        print(linje)
    print("-" * 64)


si(
    "Oppsett er klart. Samme type Keras Sequential-flyt som i modul 04.",
    f"Prosjektmappe: {PROSJEKT}",
    "",
    "Klasse-koding (bruk denne overalt):",
    "  0 = ok      godkjent støpehjul",
    "  1 = avvik   støpehjul med feil",
    "",
    "Legg egne utvalgte og merkede industribilder her FØR du kjører neste celle:",
    f"  data/ok      -> {DATA_OK}",
    f"  data/avvik   -> {DATA_AVVIK}",
    f"  field_cases  -> {FELT}  (nøyaktig 6 bilder, aldri i fit())",
)


## 2. Laste og kontrollere data (Del A)

Minst 30 bilder fra `casting_512x512/ok_front` i `data/ok/` og 30 fra `def_front` i `data/avvik/`.  
De 6 i `field_cases/` skal være **andre filer** og skal **ikke** inn i `fit()`.


In [ ]:
def list_bilder(mappe: Path):
    if not mappe.is_dir():
        raise FileNotFoundError(f"Mangler {mappe}")
    return sorted(p for p in mappe.iterdir() if p.is_file() and p.suffix.lower() in TYPER)


def til_28(sti: Path):
    # Gråskala, 28x28, verdier i [0, 1] – samme format som Fashion-MNIST i modul 04.
    g = Image.open(sti).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE))
    return np.array(g, dtype="float32") / 255.0


filer_ok = list_bilder(DATA_OK)
filer_avvik = list_bilder(DATA_AVVIK)
si(
    "Bildene er funnet i mappene.",
    f"  OK-bilder:    {len(filer_ok):3d}  i data/ok     (etikett 0)",
    f"  AVVIK-bilder: {len(filer_avvik):3d}  i data/avvik  (etikett 1)",
    f"  Til sammen:   {len(filer_ok) + len(filer_avvik):3d}  bilder til trenings- og valideringsdata",
    "",
    "Krav: minst 30 i hver klasse. Felt-casene skal IKKE ligge i data/.",
)
if len(filer_ok) < 30 or len(filer_avvik) < 30:
    raise ValueError("Minst 30 bilder i data/ok og data/avvik. Ikke bruk field_cases her.")

n_felt = len(list_bilder(FELT)) if FELT.is_dir() else 0
felt_navn = {p.name for p in list_bilder(FELT)} if FELT.is_dir() else set()
lekk = [p.name for p in filer_ok + filer_avvik if p.name in felt_navn]
if lekk:
    raise ValueError(f"Felt-case ligger også i data/: {lekk}")
if n_felt != 6:
    si(f"Merk: field_cases/ har {n_felt} bilder. Oppgaven krever nøyaktig 6.")

fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for rad, (navn, filer) in enumerate([("OK (0)", filer_ok), ("AVVIK (1)", filer_avvik)]):
    axes[rad, 0].imshow(Image.open(filer[0]), cmap="gray")
    axes[rad, 0].set_title(f"{navn} original")
    axes[rad, 0].axis("off")
    axes[rad, 1].imshow(til_28(filer[0]), cmap="gray", vmin=0, vmax=1)
    axes[rad, 1].set_title("28×28  [0,1]")
    axes[rad, 1].axis("off")
    axes[rad, 2].imshow(Image.open(filer[1] if len(filer) > 1 else filer[0]), cmap="gray")
    axes[rad, 2].set_title("ett til")
    axes[rad, 2].axis("off")
plt.tight_layout()
plt.show()
si(
    "Hva du ser i figurene over:",
    "  Venstre: originalbildet (slik du valgte det fra Kaggle).",
    "  Midten:  samme bilde som gråskala 28×28, verdier mellom 0 og 1.",
    "  Høyre:   ett bilde til fra samme klasse.",
    "",
    "Flatten endrer formen fra 28×28 piksler til en vektor med 784 pikselverdier.",
    "Pikselverdiene beholdes, men den todimensjonale bildeformen forsvinner.",
    "Små detaljer kan bli utydelige etter 28×28. Det er forventet.",
)


## 3. Train/validation-splitt

Bildene i `data/` deles her. **Modell A og Modell B skal bruke nøyaktig denne splitten.**  
`field_cases/` er **ikke** med.


In [ ]:
def last_klasse(filer, etikett):
    X = np.stack([til_28(p) for p in filer])
    y = np.full((len(filer),), etikett, dtype="int32")
    return X, y


X_ok, y_ok = last_klasse(filer_ok, 0)
X_avvik, y_avvik = last_klasse(filer_avvik, 1)
X = np.concatenate([X_ok, X_avvik])
y = np.concatenate([y_ok, y_avvik])

rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
n_val = max(1, int(0.2 * len(X)))
X_val, y_val = X[idx[:n_val]], y[idx[:n_val]]
X_train, y_train = X[idx[n_val:]], y[idx[n_val:]]

n_ok_train = int(np.sum(y_train == 0))
n_avvik_train = int(np.sum(y_train == 1))
n_ok_val = int(np.sum(y_val == 0))
n_avvik_val = int(np.sum(y_val == 1))
si(
    "Bildene i data/ er blandet og delt. field_cases/ er ikke med.",
    "",
    f"Totalt fra data/: {len(X)} bilder. Hvert bilde er {IMAGE_SIZE}×{IMAGE_SIZE}.",
    f"Form på X_train: {X_train.shape}   (antall, høyde, bredde)",
    f"Form på y_train: {y_train.shape}   (0 = OK, 1 = AVVIK)",
    "",
    f"Treningsdata (ca. 80 %):     {len(X_train)}  –  OK={n_ok_train}, AVVIK={n_avvik_train}",
    f"Valideringsdata (ca. 20 %):  {len(X_val)}  –  OK={n_ok_val}, AVVIK={n_avvik_val}",
    "",
    "Modell A og Modell B skal bruke SAMME X_train, y_train, X_val og y_val.",
    "Da blir sammenligningen rettferdig. Ikke lag en ny splitt før Modell B.",
    "De 6 felt-casene brukes først etter at Modell B er lagret og lastet inn igjen.",
)


## 4–5. Modell A – baseline, trening og evaluate (Del C, punkt 2)

Tynnere enn Modul 04: Flatten → Dense(32, ReLU) → Dense(2, softmax).  
**SGD** og `sparse_categorical_crossentropy`. Skriv ned `LR_A`.


In [ ]:
EPOCHS = 25
BATCH = 8
SKJULT_A = 32
LR_A = 0.01


def kompiler(model, lr):
    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def lag_baseline(n_skjult, lr):
    model = keras.Sequential(
        [
            layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE)),
            layers.Flatten(),
            layers.Dense(n_skjult, activation="relu"),
            layers.Dense(2, activation="softmax"),
        ]
    )
    return kompiler(model, lr)


model_a = lag_baseline(SKJULT_A, LR_A)
model_a.summary()

si(
    "Modell A (baseline) er bygd.",
    "  Lag: Input(28,28) -> Flatten -> Dense(32, ReLU) -> Dense(2, softmax)",
    "  Utgang: indeks 0 = OK, indeks 1 = AVVIK",
    f"  Optimizer: SGD med learning_rate LR_A = {LR_A}",
    "  Loss: sparse_categorical_crossentropy",
    f"  Trening: {EPOCHS} epochs, batch_size={BATCH}",
    "",
    "fit() under: loss/accuracy = treningsdata, val_* = valideringsdata.",
)

history_a = model_a.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH, shuffle=True, verbose=1,
)
loss_a, acc_a = model_a.evaluate(X_val, y_val, verbose=0)
si(
    "Modell A er ferdig trent. evaluate() på valideringsdata (ikke felt-case):",
    f"  Valideringstap (loss):     {loss_a:.3f}   (lavere = mindre feil)",
    f"  Validerings-accuracy:      {acc_a:.3f}   ({int(round(acc_a * len(y_val)))} av {len(y_val)} valideringsbilder riktig)",
    f"  Siste trenings-accuracy:   {history_a.history['accuracy'][-1]:.3f}",
    "",
    "Accuracy alene forteller ikke OM feilen er falsk OK eller falsk AVVIK.",
    "Det ser du i confusion matrix under.",
)


## 6. Loss-kurve (Modell A)


In [ ]:
def plott_tap(history, tittel, fil):
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    ax.plot(history.history["loss"], label="treningsdata")
    ax.plot(history.history["val_loss"], label="valideringsdata")
    ax.set_title(tittel)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    FIGURES.mkdir(exist_ok=True)
    fig.savefig(FIGURES / fil, dpi=120, bbox_inches="tight")
    plt.show()


plott_tap(history_a, "Modell A – baseline", "loss_baseline.png")
si(
    "Kurven over viser tap (loss) for hver epoch.",
    "  Treningsdata:     synker den? Da lærer nettverket av treningen.",
    "  Valideringsdata:  synker den også? Da generaliserer den bedre.",
    "  Skiller de lag: trening synker, val flater eller stiger = overtilpasning.",
    "",
    "TODO: skriv 2–3 setninger om kurven i feilanalysen senere.",
)


## 7. Confusion matrix (Modell A)

Fire ruter. Finn **antall falsk OK** (faktisk AVVIK → predikert OK).


In [ ]:
def vis_forvirring(y_true, y_pred, tittel):
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    ok_ok, ok_avvik = int(cm[0, 0]), int(cm[0, 1])
    avvik_ok, avvik_avvik = int(cm[1, 0]), int(cm[1, 1])
    si(
        tittel,
        f"  Faktisk OK,    predikert OK:     {ok_ok}",
        f"  Faktisk OK,    predikert AVVIK:  {ok_avvik}   <- falsk AVVIK",
        f"  Faktisk AVVIK, predikert OK:     {avvik_ok}   <- FALSK OK (dyrest her)",
        f"  Faktisk AVVIK, predikert AVVIK:  {avvik_avvik}",
        "",
        f"Antall falsk OK: {avvik_ok}. Accuracy alene viser ikke denne ruten.",
    )
    fig, ax = plt.subplots(figsize=(4.4, 3.6))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], labels=["pred OK", "pred AVVIK"])
    ax.set_yticks([0, 1], labels=["faktisk OK", "faktisk AVVIK"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=16)
    ax.set_title(tittel)
    plt.tight_layout()
    plt.show()
    return cm


pred_a = np.argmax(model_a.predict(X_val, verbose=0), axis=1)
cm_a = vis_forvirring(y_val, pred_a, "Confusion matrix – Modell A (valideringsdata)")


## 8. Hypotese (Del C, punkt 3) – skriv FØR du trener Modell B

**TODO – fyll inn, deretter kjør neste celle.** Endre nøyaktig én ting: noder, eller ett ekstra Dense-lag, eller learning_rate. Ikke Adam. Ikke CNN. Ikke «bare flere epochs».

Jeg endrer nøyaktig én ting: **______**.

Jeg forventer at … fordi …


## 9. Modell B – én endring (Del C, punkt 3)

Sett `FORBEDRING` til nøyaktig én verdi. Samme `X_train` / `X_val` som Modell A.


In [ ]:
FORBEDRING = ""  # TODO: sett "noder" eller "ekstra_lag" eller "lr"
SKJULT_B = 64
LR_B = 0.001


def lag_b():
    if FORBEDRING == "noder":
        return lag_baseline(SKJULT_B, LR_A)
    if FORBEDRING == "ekstra_lag":
        model = keras.Sequential(
            [
                layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE)),
                layers.Flatten(),
                layers.Dense(SKJULT_A, activation="relu"),
                layers.Dense(SKJULT_A, activation="relu"),
                layers.Dense(2, activation="softmax"),
            ]
        )
        return kompiler(model, LR_A)
    if FORBEDRING == "lr":
        return lag_baseline(SKJULT_A, LR_B)
    raise ValueError("FORBEDRING må være noder, ekstra_lag eller lr")


model_b = lag_b()
model_b.summary()
history_b = model_b.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH, shuffle=True, verbose=1,
)
loss_b, acc_b = model_b.evaluate(X_val, y_val, verbose=0)
si(
    f"Modell B er trent. Du endret nøyaktig én ting: {FORBEDRING}.",
    "Samme trenings- og valideringsdata som Modell A.",
    f"  Valideringstap:       {loss_b:.3f}",
    f"  Validerings-accuracy: {acc_b:.3f}",
)


## 10. Sammenligning A vs. B (Del C, punkt 4)

Samme valideringsdata. Loss-kurver og confusion matrix.


In [ ]:
plott_tap(history_b, "Modell B – én endring", "loss_forbedret.png")
pred_b = np.argmax(model_b.predict(X_val, verbose=0), axis=1)
cm_b = vis_forvirring(y_val, pred_b, "Confusion matrix – Modell B (valideringsdata)")

tr_a = history_a.history["accuracy"][-1]
tr_b = history_b.history["accuracy"][-1]
falsk_ok_a = int(cm_a[1, 0])
falsk_ok_b = int(cm_b[1, 0])

if acc_b > acc_a + 0.005:
    vinner = "B er bedre på validerings-accuracy enn A."
elif acc_a > acc_b + 0.005:
    vinner = "A er fortsatt bedre på validerings-accuracy."
else:
    vinner = "A og B er omtrent like på validerings-accuracy."

si(
    "Sammenligning på SAMME valideringsdata:",
    f"  A: loss={loss_a:.3f}  val-acc={acc_a:.3f}  trenings-acc={tr_a:.3f}  falsk OK={falsk_ok_a}",
    f"  B: loss={loss_b:.3f}  val-acc={acc_b:.3f}  trenings-acc={tr_b:.3f}  falsk OK={falsk_ok_b}",
    "",
    vinner,
    "Se også om antall falsk OK gikk ned. Det er den dyre feilen her.",
    "",
    "TODO: fyll tabellen under med dine tall.",
)


| Modell | Endring | LR | Val-acc | Falsk OK (val) | Felt-case riktig | Merknad |
|--------|---------|----|---------|----------------|------------------|---------|
| A | baseline | | | | (etter steg 13) | |
| B | | | | | | |

**TODO:** Fyll tabellen med tallene dine. Sammenlign A og B på samme valideringsdata: val-acc, loss-kurver og antall falsk OK.


## 11–12. Save og load Modell B (Del C, punkt 5)

Prediksjon på felt-case skal bruke den **innlastede** filen, ikke `model_b` i minnet etter `fit()`.


In [ ]:
MODELS.mkdir(exist_ok=True)
STI = MODELS / "quality_model.keras"
model_b.save(STI)
loaded = keras.models.load_model(STI)
si(
    "Modell B er lagret og lastet inn på nytt.",
    f"  Fil: {STI}",
    "  Felt-casene under predikeres med 'loaded', ikke model_b.",
    "  models/ skal IKKE pushes til GitHub.",
)


## 13. Felt-case med innlastet modell (Del C, punkt 6)

Fyll `FELT_CASE_FASIT` med **dine** filnavn og faktisk klasse (`ok` / `avvik`).  
Skriv «X av 6 felt-case ble klassifisert riktig» – ikke prosent-accuracy.


In [ ]:
# TODO: fyll inn DINE 6 filnavn og faktisk klasse ("ok" eller "avvik").
# Bruk nøyaktig samme filnavn som i field_cases/. Ikke kopier et ferdig fasit-sett.
FELT_CASE_FASIT = {
    "FILNAVN_1.jpg": "ok",
    "FILNAVN_2.jpg": "ok",
    "FILNAVN_3.jpg": "avvik",
    "FILNAVN_4.jpg": "avvik",
    "FILNAVN_5.jpg": "ok",
    "FILNAVN_6.jpg": "avvik",
}

felt = list_bilder(FELT)
if len(felt) != 6:
    raise ValueError(f"Trenger nøyaktig 6 bilder i {FELT}, fant {len(felt)}")

rader = []
print()
print("-" * 64)
print("Prediksjon på 6 felt-case med INNLASTET modell")
print("Softmax-score = modellens tall for valgt klasse.")
print("0.94 betyr ikke at den er 94 % sikker i produksjon.")
print("-" * 64)

for nr, p in enumerate(felt, start=1):
    x = til_28(p)
    s = loaded.predict(x[np.newaxis, ...], verbose=0)[0]
    i = int(np.argmax(s))
    pred, score = KLASSER[i], float(s[i])
    sann_rå = FELT_CASE_FASIT.get(p.name)
    if sann_rå is None:
        raise KeyError(f"Fyll FELT_CASE_FASIT for {p.name}")
    sann = str(sann_rå).strip().lower()
    if sann not in KLASSER:
        raise ValueError(f"FELT_CASE_FASIT[{p.name}] må være 'ok' eller 'avvik', fikk {sann_rå!r}")
    ok = pred == sann
    if ok:
        feiltype = "–"
        utfall = "RIKTIG"
        betydning = "Modellen og din merking er enige."
    elif sann == "avvik" and pred == "ok":
        feiltype = "falsk OK"
        utfall = "FEIL – falsk OK"
        betydning = "Et AVVIK ble klassifisert som OK. Defekt del kan slippe videre."
    else:
        feiltype = "falsk AVVIK"
        utfall = "FEIL – falsk AVVIK"
        betydning = "En OK-del ble klassifisert som AVVIK. God del ble stoppet."
    rader.append((p, sann, pred, score, ok, feiltype, float(s[0]), float(s[1])))

    print(f"{nr}. {p.name}")
    print(f"   Faktisk klasse:               {sann}")
    print(f"   Predikert klasse:             {pred}")
    print(f"   Softmax:  OK={s[0]:.3f}   AVVIK={s[1]:.3f}")
    print(f"   Score (valgt klasse):         {score:.3f}")
    print(f"   Riktig/feil:                  {utfall}")
    print(f"   Feiltype:                     {feiltype}")
    print(f"   {betydning}")
    print()

n = sum(r[4] for r in rader)
falsk_ok = sum(1 for r in rader if r[5] == "falsk OK")
falsk_avvik = sum(1 for r in rader if r[5] == "falsk AVVIK")
si(
    f"{n} av {len(rader)} felt-case ble klassifisert riktig.",
    f"  Falsk OK:    {falsk_ok}",
    f"  Falsk AVVIK: {falsk_avvik}",
    "",
    f"Skriv «{n} av {len(rader)} felt-case ble klassifisert riktig» – ikke prosent.",
)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for ax, (p, sann, pred, score, ok, feiltype, _, _) in zip(axes.ravel(), rader):
    ax.imshow(til_28(p), cmap="gray", vmin=0, vmax=1)
    farge = "tab:green" if ok else "tab:red"
    ax.set_title(f"{p.name}\n{sann}→{pred}  {score:.2f}", color=farge, fontsize=8)
    ax.axis("off")
plt.suptitle("Felt-case (innlastet Modell B)")
plt.tight_layout()
plt.show()

feil = [r for r in rader if not r[4]]
if not feil:
    si(
        "Ingen feil på de 6 felt-casene denne gangen.",
        "6 bilder er for få til å være sikker. Gå likevel gjennom Del D.",
    )
else:
    si(
        f"Modellen tok feil på {len(feil)} av {len(rader)} felt-case.",
        "Bildene under: original og 28×28. Koble forklaringen til det du ser.",
    )

for p, sann, pred, score, _, feiltype, _, _ in feil:
    print(f"Feilbilde: {p.name}   faktisk={sann}   pred={pred}   {feiltype}   score={score:.3f}")
    fig, ax = plt.subplots(1, 2, figsize=(6, 2.8))
    ax[0].imshow(Image.open(p), cmap="gray")
    ax[0].set_title("original")
    ax[0].axis("off")
    ax[1].imshow(til_28(p), cmap="gray", vmin=0, vmax=1)
    ax[1].set_title("28×28")
    ax[1].axis("off")
    fig.suptitle(f"{p.name}: {sann} → {pred} ({feiltype})")
    plt.tight_layout()
    plt.show()


## 14. Feilanalyse (Del C, punkt 6)

**TODO:** Forklar observerte feil konkret (28×28, lys, bakgrunn, merking, lite sett, forskjell mot treningsdata). Ikke skriv «den skjønte det ikke».

Skriv «X av 6 felt-case ble klassifisert riktig» – ikke prosent. Avslutt med: gir resultatene grunnlag for videre utvikling, og hva må forbedres?

Skriv her: ...


## 15. Del B – Teori

**TODO:** Egne ord, maks. ca. 250 ord.

1. Hva er forskjellen på `fit()`, `evaluate()` og `predict()`? Hva kjører du når et nytt bilde kommer inn etter at modellen er ferdigtrent?
2. Nevn to tegn på overtilpasning.
3. Forklar falsk OK versus falsk AVVIK. Hvilken er dyrest i denne støpehjul-casen, og hvorfor?

Skriv her: ...


## 16. Del D – Teknisk vurdering av PoC

**TODO:** Ca. 200–300 ord. Dette er en proof-of-concept, ikke et ferdig system.

1. Hva PoC-en viste. Ta med at du selv lastet ned og valgte ut bildene fra Kaggle (`casting_512x512`).
2. Hva resultatene ikke viser sikkert (lite datasett, enkel modell, bare 6 field_cases, 28×28).
3. Hvordan det gikk på de 6 field_cases («X av 6»).
4. Om falsk OK forekom (faktisk AVVIK predikert som OK).
5. Om resultatene gir grunnlag for videre utvikling.
6. Hva du ville gjort videre: mer eller bedre data, eller endringer i modellen?
7. Kort om eventuell KI-bruk (hva du brukte KI til, og ett sted du sjekket selv).

Skriv her: ...

---

**Levering:** privat GitHub-repo + lenke i Teams (Arbeidskrav 1). Frist: **11. oktober 2026**.  
I repoet: denne notebooken (kjørt), `data/ok`, `data/avvik`, `field_cases` og `figures/` (tap-kurver).  
Ikke push `.venv`, `archive.zip`, hele Kaggle-settet eller `models/`.
